# 04 — Hotspot Evolution

This notebook documents the temporal hotspot classification derived from annual binary detection rasters for **2015–2025**.

The classification is rule-based and follows the project's hotspot definitions.

In [ ]:
import pandas as pd

rules = pd.DataFrame({
    "Class": ["Persistent", "Emerging", "Abandoned"],
    "Definition": [
        "Active in >=8 of 11 years",
        "No activity in 2015–2019 and active in >=2 years during 2020–2025",
        "Active in >=2 years during 2015–2019 and no activity during 2022–2025",
    ],
    "Raster code": [1, 2, 3],
})

display(rules)

## Classification logic

For each pixel, the annual binary detections are evaluated across the three temporal windows:

- **2015–2019:** early period
- **2020–2025:** later period
- **2022–2025:** recent period

The resulting combined raster uses:

```text
0 = no hotspot
1 = persistent
2 = emerging
3 = abandoned
```

Where categories overlap, the project workflow gives **persistent** status priority.

In [ ]:
# A small, transparent implementation of the documented rules.
# `annual_binary` should be a NumPy array shaped:
# (11 years, rows, columns), with 1 = active and 0 = inactive.

import numpy as np

def classify_hotspots(annual_binary):
    annual_binary = np.asarray(annual_binary)
    if annual_binary.shape[0] != 11:
        raise ValueError("Expected 11 annual rasters for 2015–2025.")

    early = annual_binary[0:5]
    late = annual_binary[5:11]
    recent = annual_binary[7:11]

    early_count = early.sum(axis=0)
    late_count = late.sum(axis=0)
    recent_count = recent.sum(axis=0)

    persistent = annual_binary.sum(axis=0) >= 8
    emerging = (early_count == 0) & (late_count >= 2)
    abandoned = (early_count >= 2) & (recent_count == 0)

    combined = np.zeros(annual_binary.shape[1:], dtype=np.uint8)
    combined[abandoned] = 3
    combined[emerging] = 2
    combined[persistent] = 1

    return persistent, emerging, abandoned, combined

## Spatial output

The production script `hotspot_evolution.py` reads the annual GeoTIFFs, validates that the 11 rasters are spatially compatible, applies the rules above, and writes separate and combined hotspot rasters.

The cleaned script also checks:

- that all 11 years are represented,
- that years are not duplicated,
- that raster dimensions match,
- that CRS values match,
- and that affine transforms match.

This prevents a temporal classification from silently combining spatially incompatible rasters.

## Interpretation

The hotspot classes describe **temporal behaviour in the model-derived detection record**:

- Persistent areas show repeated detections over most of the study period.
- Emerging areas appear after the early period and recur in the later period.
- Abandoned areas were active in the early period but have no detections in the recent period.

These categories are analytical classifications rather than direct field-confirmed land-use histories.